#### 📚 Importación de librerías


In [ ]:
# Librerias
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


#### ⏫ Importacion de datos


In [2]:
BASE_DIR = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

interim_files = sorted(INTERIM_DIR.glob("spotify_tracks_interim_*.parquet"), key=os.path.getmtime)
if not interim_files:
    raise FileNotFoundError("No se encontraron archivos interim en data/interim")

latest_interim_path = interim_files[-1]
print("Usando INTERIM:", latest_interim_path)

df = pd.read_parquet(latest_interim_path)
print("Shape inicial:", df.shape)

df.head(3)


Usando INTERIM: /Users/nicoguisande/Library/CloudStorage/OneDrive-Personal/Nicolas/Personal/Cursos/Carrera - Data Scientist - Coder House/Curso - Data Science II/Proyecto Final/spotify-growth-ml/data/interim/spotify_tracks_interim_20260301_000321.parquet


Shape inicial: (66156, 26)


,track_id,track_name,album_id,album_name,artist_id,artist_name,playlist_id,playlist_name,track_popularity,duration_ms,explicit,album_release_date,added_at,artist_avg_popularity,playlist_avg_popularity,playlist_popularity_std,artist_popularity_std,years_since_release,album_release_year,playlist_track_count,artist_track_count,artist_avg_duration,playlist_avg_duration,popular_high,popular_high_p50,popular_high_p60
0,7Dev2VJdJKwIIUdj3lcQl9,Release The Pressure,3mFlg0trNYSCnTd0RcN0wx,Release The Pressure,7CajNmpbOovFoOoasH2HaY,Calvin Harris,2237sMNMlXS4wWLgdQ1UuV,Workout Motivation 2026 💪,73,203668,False,2026-02-06,2026-02-23T13:55:47Z,53.585714,53.695279,19.799138,21.772535,0.062971,2026.0,233.0,70,3.651259,2.59103,1,1,1
1,09CnYHiZ5jGT1wr1TXJ9Zt,Thank You (Not So Bad),1J7XItLnNLegigdh4AjGKN,Thank You (Not So Bad),73jBynjsVtofjRpdpRAJGk,Dimitri Vegas & Like Mike,2237sMNMlXS4wWLgdQ1UuV,Workout Motivation 2026 💪,82,140000,False,2023-12-01,2026-02-23T13:55:47Z,26.609756,53.695279,19.799138,25.557658,2.247775,2023.0,233.0,41,3.695037,2.59103,1,1,1
2,2qlqMa0e422LZyGw1J5for,YEAH,3reHnBJbOMrlymwkwYqtEH,YEAH,73jBynjsVtofjRpdpRAJGk,Dimitri Vegas & Like Mike,2237sMNMlXS4wWLgdQ1UuV,Workout Motivation 2026 💪,69,141551,True,2025-03-14,2026-02-23T13:55:47Z,26.609756,53.695279,19.799138,25.557658,0.963723,2025.0,233.0,41,3.695037,2.59103,1,1,1


#### 🛡️ Validación de contrato de entrada


In [3]:
TARGET_Q = 0.70

required_columns = [
    "track_id", "track_popularity", "duration_ms", "popular_high",
    "artist_avg_popularity", "playlist_avg_popularity", "playlist_popularity_std",
    "artist_popularity_std", "years_since_release", "album_release_year",
    "playlist_track_count", "artist_track_count", "artist_avg_duration", "playlist_avg_duration"
]

missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise KeyError(f"Faltan columnas requeridas en interim: {missing_required}")

threshold_from_quantile = float(df["track_popularity"].quantile(TARGET_Q))
threshold_from_target = float(df.loc[df["popular_high"] == 1, "track_popularity"].min())

if abs(threshold_from_quantile - threshold_from_target) > 1e-9:
    raise ValueError(
        f"Inconsistencia target P70. Quantile={threshold_from_quantile}, target_min={threshold_from_target}"
    )

target_before_counts = df["popular_high"].value_counts().sort_index()
target_before_pct = (df["popular_high"].value_counts(normalize=True).sort_index() * 100).round(2)

print(f"Contrato OK. Umbral P70 validado: {threshold_from_quantile:.1f}")
print("Distribución target (antes de filtro):", target_before_counts.to_dict())
print("Distribución target % (antes de filtro):", target_before_pct.to_dict())


Contrato OK. Umbral P70 validado: 51.0
Distribución target (antes de filtro): {0: 45560, 1: 20596}
Distribución target % (antes de filtro): {0: 68.87, 1: 31.13}


#### 🔎 Filtro de población por duración


In [4]:
df["duration_min"] = df["duration_ms"] / 60000

rows_before = len(df)
df = df[(df["duration_min"] >= 0.5) & (df["duration_min"] <= 15)].copy()
rows_after = len(df)
rows_removed = rows_before - rows_after
rows_removed_pct = (rows_removed / rows_before) * 100 if rows_before else 0

target_after_counts = df["popular_high"].value_counts().sort_index()
target_after_pct = (df["popular_high"].value_counts(normalize=True).sort_index() * 100).round(2)

print(f"Filas antes de filtro: {rows_before:,}")
print(f"Filas después de filtro: {rows_after:,}")
print(f"Filas removidas: {rows_removed:,} ({rows_removed_pct:.2f}%)")
print("Distribución target (después de filtro):", target_after_counts.to_dict())
print("Distribución target % (después de filtro):", target_after_pct.to_dict())


Filas antes de filtro: 66,156
Filas después de filtro: 66,074
Filas removidas: 82 (0.12%)
Distribución target (después de filtro): {0: 45479, 1: 20595}
Distribución target % (después de filtro): {0: 68.83, 1: 31.17}


#### 🎯 Selección de features finales


In [5]:
final_features = [
    "artist_avg_popularity",
    "playlist_avg_popularity",
    "playlist_popularity_std",
    "artist_popularity_std",
    "years_since_release",
    "album_release_year",
    "playlist_track_count",
    "artist_track_count",
    "artist_avg_duration",
    "playlist_avg_duration",
]

print(f"Total features finales: {len(final_features)}")
print(final_features)


Total features finales: 10
['artist_avg_popularity', 'playlist_avg_popularity', 'playlist_popularity_std', 'artist_popularity_std', 'years_since_release', 'album_release_year', 'playlist_track_count', 'artist_track_count', 'artist_avg_duration', 'playlist_avg_duration']


#### 🧹 Imputación explícita por mediana


In [6]:
missing_before = df[final_features].isna().sum().sort_values(ascending=False)
missing_before_total = int(missing_before.sum())

imputation_values = {}
for col in final_features:
    median_value = float(df[col].median())
    imputation_values[col] = median_value
    df[col] = df[col].fillna(median_value)

missing_after = df[final_features].isna().sum().sort_values(ascending=False)
missing_after_total = int(missing_after.sum())
resolved_missing_pct = ((missing_before_total - missing_after_total) / missing_before_total * 100) if missing_before_total > 0 else 100.0

imputation_report = pd.DataFrame({
    "feature": final_features,
    "impute_value_median": [imputation_values[f] for f in final_features],
    "missing_before": [int(missing_before[f]) for f in final_features],
    "missing_after": [int(missing_after[f]) for f in final_features],
})

print(f"Nulos totales antes de imputación: {missing_before_total:,}")
print(f"Nulos totales después de imputación: {missing_after_total:,}")
print(f"Porcentaje de nulos resueltos: {resolved_missing_pct:.2f}%")

imputation_report.sort_values("missing_before", ascending=False)


Nulos totales antes de imputación: 27,141
Nulos totales después de imputación: 0
Porcentaje de nulos resueltos: 100.00%


,feature,impute_value_median,missing_before,missing_after
3,artist_popularity_std,14.138768,11015,0
1,playlist_avg_popularity,35.808511,4027,0
9,playlist_avg_duration,3.359414,4027,0
2,playlist_popularity_std,19.321796,4027,0
6,playlist_track_count,209.000000,4027,0
5,album_release_year,2021.000000,9,0
4,years_since_release,4.470910,9,0
0,artist_avg_popularity,37.600000,0,0
7,artist_track_count,9.000000,0,0
8,artist_avg_duration,3.301767,0,0


#### 🧱 Armado de dataset model-ready

In [7]:
output_columns = ["track_id", "track_popularity"] + final_features + ["popular_high"]
model_df = df[output_columns].copy()

if model_df[final_features].isna().sum().sum() != 0:
    raise ValueError("Persisten nulos en features finales después de imputación")

print("Shape model_df:", model_df.shape)
print("Columnas model_df:", model_df.columns.tolist())
model_df.head(3)


Shape model_df: (66074, 13)
Columnas model_df: ['track_id', 'track_popularity', 'artist_avg_popularity', 'playlist_avg_popularity', 'playlist_popularity_std', 'artist_popularity_std', 'years_since_release', 'album_release_year', 'playlist_track_count', 'artist_track_count', 'artist_avg_duration', 'playlist_avg_duration', 'popular_high']


,track_id,track_popularity,artist_avg_popularity,playlist_avg_popularity,playlist_popularity_std,artist_popularity_std,years_since_release,album_release_year,playlist_track_count,artist_track_count,artist_avg_duration,playlist_avg_duration,popular_high
0,7Dev2VJdJKwIIUdj3lcQl9,73,53.585714,53.695279,19.799138,21.772535,0.062971,2026.0,233.0,70,3.651259,2.59103,1
1,09CnYHiZ5jGT1wr1TXJ9Zt,82,26.609756,53.695279,19.799138,25.557658,2.247775,2023.0,233.0,41,3.695037,2.59103,1
2,2qlqMa0e422LZyGw1J5for,69,26.609756,53.695279,19.799138,25.557658,0.963723,2025.0,233.0,41,3.695037,2.59103,1


### 💾 Guardado del dataset procesado


In [8]:
ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
out_path = PROCESSED_DIR / f"spotify_model_ready_{ts}.parquet"

model_df.to_parquet(out_path, index=False)

print("Archivo generado:", out_path)
print(f"Tamaño: {out_path.stat().st_size / 1024**2:.2f} MB")
print("Shape exportada:", model_df.shape)


Archivo generado: /Users/nicoguisande/Library/CloudStorage/OneDrive-Personal/Nicolas/Personal/Cursos/Carrera - Data Scientist - Coder House/Curso - Data Science II/Proyecto Final/spotify-growth-ml/data/processed/spotify_model_ready_20260301_005928.parquet
Tamaño: 2.40 MB
Shape exportada: (66074, 13)


#### ✅ Check de compatibilidad con `03_modeling`


In [9]:
X = model_df.drop(columns=["track_id", "track_popularity", "popular_high"])
y = model_df["popular_high"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Compatibilidad OK con 03_modeling")
print("X shape:", X.shape)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y distribution:", y.value_counts(normalize=True).round(4).to_dict())


Compatibilidad OK con 03_modeling
X shape: (66074, 10)
X_train: (52859, 10) | X_test: (13215, 10)
y distribution: {0: 0.6883, 1: 0.3117}


### 📈 Resumen Ejecutivo


In [10]:
print("=" * 80)
print("RESUMEN EJECUTIVO - FEATURE ENGINEERING")
print("=" * 80)

print("\n📥 ENTRADA:")
print(f"  • Archivo interim: {latest_interim_path.name}")
print(f"  • Shape inicial: {rows_before:,} filas")

print("\n🔎 FILTRO DE POBLACIÓN:")
print("  • Regla: 0.5 <= duration_min <= 15")
print(f"  • Filas removidas: {rows_removed:,} ({rows_removed_pct:.2f}%)")
print(f"  • Shape post-filtro: {rows_after:,} filas")

print("\n🎯 TARGET (popular_high, P70 heredado):")
print(f"  • Umbral validado: {threshold_from_quantile:.1f}")
print(f"  • Distribución antes: {target_before_counts.to_dict()} | %: {target_before_pct.to_dict()}")
print(f"  • Distribución después: {target_after_counts.to_dict()} | %: {target_after_pct.to_dict()}")

print("\n🧹 CALIDAD DE FEATURES FINALES:")
print(f"  • Features finales: {len(final_features)}")
print(f"  • Nulos antes imputación: {missing_before_total:,}")
print(f"  • Nulos después imputación: {missing_after_total:,}")
print(f"  • Nulos resueltos: {resolved_missing_pct:.2f}%")

print("\n🧱 SALIDA MODEL-READY:")
print(f"  • Columnas exactas: {len(output_columns)}")
print(f"  • Shape final: {model_df.shape[0]:,} x {model_df.shape[1]}")
print(f"  • Archivo procesado: {out_path.name}")

print("\n📋 LISTA DE FEATURES FINALES:")
for feat in final_features:
    print(f"  - {feat}")

print("\n✅ COMPATIBILIDAD 03_MODELING:")
print(f"  • X usable shape: {X.shape}")
print("  • split estratificado: OK")
print("=" * 80)


RESUMEN EJECUTIVO - FEATURE ENGINEERING

📥 ENTRADA:
  • Archivo interim: spotify_tracks_interim_20260301_000321.parquet
  • Shape inicial: 66,156 filas

🔎 FILTRO DE POBLACIÓN:
  • Regla: 0.5 <= duration_min <= 15
  • Filas removidas: 82 (0.12%)
  • Shape post-filtro: 66,074 filas

🎯 TARGET (popular_high, P70 heredado):
  • Umbral validado: 51.0
  • Distribución antes: {0: 45560, 1: 20596} | %: {0: 68.87, 1: 31.13}
  • Distribución después: {0: 45479, 1: 20595} | %: {0: 68.83, 1: 31.17}

🧹 CALIDAD DE FEATURES FINALES:
  • Features finales: 10
  • Nulos antes imputación: 27,141
  • Nulos después imputación: 0
  • Nulos resueltos: 100.00%

🧱 SALIDA MODEL-READY:
  • Columnas exactas: 13
  • Shape final: 66,074 x 13
  • Archivo procesado: spotify_model_ready_20260301_005928.parquet

📋 LISTA DE FEATURES FINALES:
  - artist_avg_popularity
  - playlist_avg_popularity
  - playlist_popularity_std
  - artist_popularity_std
  - years_since_release
  - album_release_year
  - playlist_track_count
  